# GP4 ReAct-IR Qwen2.5 QLoRA Cloud Workflow

Colab runtime must persist generated data, reports, checkpoints, adapters, and inference outputs to Google Drive only.

The notebook uses `gp4_finetune_factory_source_bundle.zip` from `$CLOUD_ROOT/bundles/` when present. Otherwise it clones the pushed `codex/gp4-react-ir-cloud-workflow` branch into the ephemeral runtime. Generated data, reports, checkpoints, adapters, inference outputs, and packages stay under `CLOUD_ROOT`.


In [ ]:
import json
import os
import shutil
import subprocess
import zipfile
from datetime import datetime
from pathlib import Path
from google.colab import drive

def mount_google_drive_or_explain():
    try:
        drive.mount('/content/drive')
    except ValueError as exc:
        raise RuntimeError('Google Drive mount failed. Approve the Google Drive permission prompt in the open OAuth tab, then rerun this setup cell.') from exc

mount_google_drive_or_explain()
DRIVE_ACCOUNT_EMAIL = 'johnwickiller4444@gmail.com'
DRIVE_ROOT = os.environ.get('GP4_DRIVE_ROOT', '/content/drive/MyDrive/gp4_finetune_factory')
SOURCE_BRANCH = os.environ.get('GP4_SOURCE_BRANCH', 'codex/gp4-react-ir-cloud-workflow')
FACTORY_SOURCE_EXPECTED_COMMIT = os.environ.get('GP4_FACTORY_SOURCE_EXPECTED_COMMIT', '').strip()
if not FACTORY_SOURCE_EXPECTED_COMMIT:
    resolved_source = subprocess.run(['git', 'ls-remote', 'https://github.com/Hieu-RMX18/gp4_finetune_factory.git', f'refs/heads/{SOURCE_BRANCH}'], text=True, capture_output=True, check=True).stdout.strip().split()
    if not resolved_source:
        raise RuntimeError(f'Unable to resolve source branch commit: {SOURCE_BRANCH}')
    FACTORY_SOURCE_EXPECTED_COMMIT = resolved_source[0]
GP4_WS_REPO_URL = os.environ.get('GP4_WS_REPO_URL', 'https://github.com/Hieu-RMX18/gp4_ws.git')
GP4_WS_BRANCH = os.environ.get('GP4_WS_BRANCH', 'ws-deep-rebuild-3526')
RUN_ID = os.environ.get('RUN_ID') or 'gp4-react-v2-300k-' + datetime.utcnow().strftime('%Y%m%d-%H%M')
CLOUD_ROOT = os.environ.get('CLOUD_ROOT') or f'{DRIVE_ROOT}/{RUN_ID}'
PREVIOUS_RUN_ID = os.environ.get('GP4_PREVIOUS_RUN_ID', '').strip()
OLD_DATASET = os.environ.get('GP4_OLD_DATASET', '').strip()
PREVIOUS_ADAPTER = os.environ.get('GP4_PREVIOUS_ADAPTER', '').strip()
def _adapter_artifact_exists(adapter_dir):
    adapter_dir = Path(adapter_dir)
    return (adapter_dir / 'adapter_config.json').is_file() and any(path.is_file() and path.stat().st_size > 0 for pattern in ('*.safetensors', '*.bin') for path in adapter_dir.glob(pattern))
def _adapter_candidates_for_run(previous_dir):
    adapter_root = Path(previous_dir) / 'models/qwen25_gp4_lora'
    candidates = []
    if _adapter_artifact_exists(adapter_root):
        candidates.append(adapter_root)
    if adapter_root.exists():
        candidates.extend(sorted((path for path in adapter_root.glob('checkpoint-*') if _adapter_artifact_exists(path)), key=lambda path: path.stat().st_mtime, reverse=True))
    return candidates
if Path(DRIVE_ROOT).exists():
    if not PREVIOUS_RUN_ID and not OLD_DATASET:
        previous_dataset_candidates = []
        for previous_dir in Path(DRIVE_ROOT).iterdir():
            if not previous_dir.is_dir():
                continue
            if previous_dir.name == RUN_ID:
                continue
            accepted_path = previous_dir / 'data/validated/accepted_300k.jsonl'
            if accepted_path.exists():
                previous_dataset_candidates.append((accepted_path.stat().st_mtime, previous_dir.name))
        if previous_dataset_candidates:
            PREVIOUS_RUN_ID = max(previous_dataset_candidates)[1]
    if not PREVIOUS_RUN_ID and not PREVIOUS_ADAPTER:
        previous_adapter_candidates = []
        for previous_dir in Path(DRIVE_ROOT).iterdir():
            if not previous_dir.is_dir():
                continue
            if previous_dir.name == RUN_ID:
                continue
            for adapter_candidate in _adapter_candidates_for_run(previous_dir):
                previous_adapter_candidates.append((adapter_candidate.stat().st_mtime, previous_dir.name, str(adapter_candidate)))
        if previous_adapter_candidates:
            _, PREVIOUS_RUN_ID, PREVIOUS_ADAPTER = max(previous_adapter_candidates)
if not OLD_DATASET and PREVIOUS_RUN_ID:
    previous_candidate = Path(DRIVE_ROOT) / PREVIOUS_RUN_ID / 'data/validated/accepted_300k.jsonl'
    if previous_candidate.exists():
        OLD_DATASET = str(previous_candidate)
if not PREVIOUS_ADAPTER and PREVIOUS_RUN_ID:
    previous_adapter_candidates = _adapter_candidates_for_run(Path(DRIVE_ROOT) / PREVIOUS_RUN_ID)
    if previous_adapter_candidates:
        PREVIOUS_ADAPTER = str(previous_adapter_candidates[0])
SOURCE_BUNDLE = f'{CLOUD_ROOT}/bundles/gp4_finetune_factory_source_bundle.zip'
WORK_DIR = Path('/content/gp4_finetune_factory_source')
EXPECTED_WORK_DIR = Path('/content/gp4_finetune_factory_source')
assert CLOUD_ROOT.startswith('/content/drive/'), 'CLOUD_ROOT must live in Google Drive.'
assert SOURCE_BUNDLE.startswith('/content/drive/'), 'Source bundle must live in Google Drive.'
if not FACTORY_SOURCE_EXPECTED_COMMIT:
    raise RuntimeError('GP4_FACTORY_SOURCE_EXPECTED_COMMIT could not be resolved before cloning or unpacking the factory source')
if len(FACTORY_SOURCE_EXPECTED_COMMIT) < 12:
    raise RuntimeError('GP4_FACTORY_SOURCE_EXPECTED_COMMIT must be a full SHA or at least 12 hex characters')
os.makedirs(f'{CLOUD_ROOT}/reports', exist_ok=True)
os.makedirs(f'{CLOUD_ROOT}/manifests', exist_ok=True)
Path(f'{CLOUD_ROOT}/manifests/drive_account_hint.txt').write_text(DRIVE_ACCOUNT_EMAIL + '\n', encoding='utf-8')
CONFIRMED_DRIVE_ACCOUNT_EMAIL = os.environ.get('GP4_DRIVE_ACCOUNT_CONFIRMED', '').strip() or DRIVE_ACCOUNT_EMAIL
if CONFIRMED_DRIVE_ACCOUNT_EMAIL != DRIVE_ACCOUNT_EMAIL:
    raise RuntimeError(f'Google Drive account confirmation mismatch: expected {DRIVE_ACCOUNT_EMAIL}, got {CONFIRMED_DRIVE_ACCOUNT_EMAIL}')
Path(f'{CLOUD_ROOT}/manifests/drive_account_confirmation.json').write_text(json.dumps({'confirmed': True, 'confirmed_email': CONFIRMED_DRIVE_ACCOUNT_EMAIL, 'expected_email': DRIVE_ACCOUNT_EMAIL, 'method': 'operator_or_explicit_requested_account_after_drive_mount'}, sort_keys=True) + '\n', encoding='utf-8')
os.chdir('/content')
if WORK_DIR.exists():
    if WORK_DIR.resolve(strict=False) != EXPECTED_WORK_DIR:
        raise RuntimeError(f'Unsafe WORK_DIR cleanup path: {WORK_DIR}')
    shutil.rmtree(WORK_DIR)
if Path(SOURCE_BUNDLE).exists():
    with zipfile.ZipFile(SOURCE_BUNDLE) as archive:
        work_dir_root = WORK_DIR.resolve(strict=False)
        for member in archive.infolist():
            member_path = (WORK_DIR / member.filename).resolve(strict=False)
            if member_path != work_dir_root and work_dir_root not in member_path.parents:
                raise RuntimeError(f'Unsafe source bundle path: {member.filename}')
        for member in archive.infolist():
            archive.extract(member, WORK_DIR)
else:
    subprocess.run(['git', 'clone', '--branch', SOURCE_BRANCH, 'https://github.com/Hieu-RMX18/gp4_finetune_factory.git', str(WORK_DIR)], check=True)
    subprocess.run(['git', '-C', str(WORK_DIR), 'checkout', '--detach', FACTORY_SOURCE_EXPECTED_COMMIT], check=True)
os.chdir(WORK_DIR)
if (WORK_DIR / '.git').exists():
    ACTUAL_FACTORY_SOURCE_COMMIT = subprocess.run(['git', '-C', str(WORK_DIR), 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
    FACTORY_SOURCE_VERIFICATION = 'git'
else:
    SOURCE_REVISION_MANIFEST = WORK_DIR / 'source_revision.json'
    if not SOURCE_REVISION_MANIFEST.exists():
        raise RuntimeError('Source bundle must include source_revision.json with factory_source_commit')
    source_revision = json.loads(SOURCE_REVISION_MANIFEST.read_text(encoding='utf-8'))
    ACTUAL_FACTORY_SOURCE_COMMIT = str(source_revision.get('factory_source_commit') or '').strip()
    FACTORY_SOURCE_VERIFICATION = 'source_revision.json'
if not ACTUAL_FACTORY_SOURCE_COMMIT.lower().startswith(FACTORY_SOURCE_EXPECTED_COMMIT.lower()):
    raise RuntimeError(f'Factory source commit mismatch: expected {FACTORY_SOURCE_EXPECTED_COMMIT}, got {ACTUAL_FACTORY_SOURCE_COMMIT}')
Path(f'{CLOUD_ROOT}/manifests/factory_source_revision.json').write_text(json.dumps({'expected_commit': FACTORY_SOURCE_EXPECTED_COMMIT, 'actual_commit': ACTUAL_FACTORY_SOURCE_COMMIT, 'source_branch': SOURCE_BRANCH, 'source_bundle': SOURCE_BUNDLE, 'verification_method': FACTORY_SOURCE_VERIFICATION}, sort_keys=True) + '\n', encoding='utf-8')
NOTEBOOK_SOURCE_PATH = WORK_DIR / 'notebooks/colab_gp4_react_qwen25_qlora.ipynb'
NOTEBOOK_DRIVE_DIR = Path(CLOUD_ROOT) / 'notebooks'
NOTEBOOK_DRIVE_COPY = NOTEBOOK_DRIVE_DIR / 'colab_gp4_react_qwen25_qlora.ipynb'
NOTEBOOK_DRIVE_DIR.mkdir(parents=True, exist_ok=True)
shutil.copy2(NOTEBOOK_SOURCE_PATH, NOTEBOOK_DRIVE_COPY)
Path(f'{CLOUD_ROOT}/manifests/colab_notebook_copy.json').write_text(json.dumps({'notebook_drive_copy': str(NOTEBOOK_DRIVE_COPY), 'source_path': str(NOTEBOOK_SOURCE_PATH), 'factory_source_commit': ACTUAL_FACTORY_SOURCE_COMMIT}, sort_keys=True) + '\n', encoding='utf-8')
SEED_PATH = str(WORK_DIR / 'data/seed/gp4_seed_starter.jsonl')
SOURCE_PLAN = str(WORK_DIR / 'docs/superpowers/plans/2026-05-20-gp4-v2-300k-readiness.md')
os.environ['RUN_ID'] = RUN_ID
os.environ['CLOUD_ROOT'] = CLOUD_ROOT
os.environ['SEED_PATH'] = SEED_PATH
os.environ['SOURCE_PLAN'] = SOURCE_PLAN
os.environ['GP4_FACTORY_SOURCE_EXPECTED_COMMIT'] = FACTORY_SOURCE_EXPECTED_COMMIT
GP4_WS_EXPECTED_COMMIT = os.environ.get('GP4_WS_EXPECTED_COMMIT', '3bbcb0726a4c3305c086c93e2b1e4a320471090b').strip()
if not GP4_WS_EXPECTED_COMMIT:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT is required before cloning or reusing the gp4_ws contract snapshot')
if len(GP4_WS_EXPECTED_COMMIT) < 12:
    raise RuntimeError('GP4_WS_EXPECTED_COMMIT must be a full SHA or at least 12 hex characters')
DRIVE_ROOT_PATH = Path(DRIVE_ROOT).resolve(strict=False)
if OLD_DATASET:
    OLD_DATASET_PATH = Path(OLD_DATASET).resolve(strict=False)
    if OLD_DATASET_PATH != DRIVE_ROOT_PATH and DRIVE_ROOT_PATH not in OLD_DATASET_PATH.parents:
        raise RuntimeError('GP4_OLD_DATASET must live under the configured Google Drive root')
    os.environ['GP4_OLD_DATASET'] = str(OLD_DATASET_PATH)
else:
    os.environ.pop('GP4_OLD_DATASET', None)
if not PREVIOUS_ADAPTER:
    raise RuntimeError('Set GP4_PREVIOUS_ADAPTER or GP4_PREVIOUS_RUN_ID to reuse a previous fine-tune adapter from Google Drive')
PREVIOUS_ADAPTER_PATH = Path(PREVIOUS_ADAPTER).resolve(strict=False)
if PREVIOUS_ADAPTER_PATH != DRIVE_ROOT_PATH and DRIVE_ROOT_PATH not in PREVIOUS_ADAPTER_PATH.parents:
    raise RuntimeError('GP4_PREVIOUS_ADAPTER must live under the configured Google Drive root')
os.environ['GP4_PREVIOUS_ADAPTER'] = str(PREVIOUS_ADAPTER_PATH)
def git_stdout(repo_path, *args):
    result = subprocess.run(['git', '-C', str(repo_path), *args], text=True, capture_output=True, check=False)
    return result.stdout.strip() if result.returncode == 0 else ''
def git_returncode(repo_path, *args):
    return subprocess.run(['git', '-C', str(repo_path), *args], text=True, capture_output=True, check=False).returncode
def configure_gp4_ws_git_for_drive(gp4_ws_path):
    if (gp4_ws_path / '.git').exists():
        subprocess.run(['git', '-C', str(gp4_ws_path), 'config', 'core.fileMode', 'false'], check=True)
        subprocess.run(['git', '-C', str(gp4_ws_path), 'update-index', '--refresh'], check=False)
def gp4_ws_snapshot_has_real_changes(gp4_ws_path):
    if git_returncode(gp4_ws_path, 'diff', '--quiet') != 0:
        return True
    if git_returncode(gp4_ws_path, 'diff', '--cached', '--quiet') != 0:
        return True
    return bool(git_stdout(gp4_ws_path, 'ls-files', '--others', '--exclude-standard'))
def gp4_ws_snapshot_needs_refresh(gp4_ws_path):
    if not gp4_ws_path.exists():
        return True
    if not (gp4_ws_path / '.git').exists():
        return True
    head = git_stdout(gp4_ws_path, 'rev-parse', 'HEAD')
    return gp4_ws_snapshot_has_real_changes(gp4_ws_path) or not head.lower().startswith(GP4_WS_EXPECTED_COMMIT.lower())
def move_dirty_gp4_ws_snapshot(gp4_ws_path):
    backup_root = GP4_WS_ROOT / 'dirty_gp4_ws_snapshots'
    backup_root.mkdir(parents=True, exist_ok=True)
    backup_path = backup_root / (gp4_ws_path.name + '_' + datetime.utcnow().strftime('%Y%m%d-%H%M%S'))
    shutil.move(str(gp4_ws_path), str(backup_path))
    print('Moved dirty/stale GP4_WS snapshot to:', backup_path)
GP4_WS_ROOT = (Path(CLOUD_ROOT) / 'contract_snapshots').resolve(strict=False)
GP4_WS = os.environ.get('GP4_WS', '').strip() or f'{CLOUD_ROOT}/contract_snapshots/gp4_ws_{GP4_WS_BRANCH}'
GP4_WS_PATH = Path(GP4_WS).resolve(strict=False)
if GP4_WS_PATH != GP4_WS_ROOT and GP4_WS_ROOT not in GP4_WS_PATH.parents:
    raise RuntimeError('GP4_WS must live under CLOUD_ROOT/contract_snapshots')
configure_gp4_ws_git_for_drive(GP4_WS_PATH)
if gp4_ws_snapshot_needs_refresh(GP4_WS_PATH):
    if GP4_WS_PATH.exists():
        move_dirty_gp4_ws_snapshot(GP4_WS_PATH)
    Path(GP4_WS).parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['git', 'clone', '--branch', GP4_WS_BRANCH, '--depth', '1', GP4_WS_REPO_URL, GP4_WS], check=True)
    configure_gp4_ws_git_for_drive(GP4_WS_PATH)
GP4_WS = str(GP4_WS_PATH)
os.environ['GP4_WS'] = GP4_WS
ACTUAL_GP4_WS_COMMIT = subprocess.run(['git', '-C', GP4_WS, 'rev-parse', 'HEAD'], text=True, capture_output=True, check=True).stdout.strip()
if not ACTUAL_GP4_WS_COMMIT.lower().startswith(GP4_WS_EXPECTED_COMMIT.lower()):
    raise RuntimeError(f'GP4_WS commit mismatch: expected {GP4_WS_EXPECTED_COMMIT}, got {ACTUAL_GP4_WS_COMMIT}')
os.environ['GP4_WS_EXPECTED_COMMIT'] = GP4_WS_EXPECTED_COMMIT
def run_checked(label, command):
    result = subprocess.run(command, text=True, capture_output=True, check=False)
    if result.stdout:
        print(result.stdout[-4000:])
    if result.stderr:
        print(result.stderr[-4000:])
    if result.returncode == 0:
        return result
    failure_report = Path(CLOUD_ROOT) / 'reports' / f'{label}_failure_{RUN_ID}.json'
    failure_report.write_text(json.dumps({'passed': False, 'label': label, 'command': command, 'returncode': result.returncode, 'stdout_tail': result.stdout[-12000:], 'stderr_tail': result.stderr[-12000:]}, indent=2, sort_keys=True) + '\n', encoding='utf-8')
    print(f'failure_report={failure_report}')
    raise subprocess.CalledProcessError(result.returncode, command, output=result.stdout, stderr=result.stderr)
print(CLOUD_ROOT)
print('Drive account hint:', DRIVE_ACCOUNT_EMAIL)
print('Factory source expected commit:', FACTORY_SOURCE_EXPECTED_COMMIT)
print('Factory source actual commit:', ACTUAL_FACTORY_SOURCE_COMMIT[:12])
print('GP4_WS contract snapshot:', os.environ['GP4_WS'])
print('GP4_WS expected commit:', os.environ['GP4_WS_EXPECTED_COMMIT'])
print('GP4_WS actual commit:', ACTUAL_GP4_WS_COMMIT[:12])
print('Old dataset:', os.environ.get('GP4_OLD_DATASET', '<adapter-only reuse; generate full 300k>'))
print('Previous adapter:', os.environ.get('GP4_PREVIOUS_ADAPTER', '<none>'))
print('Notebook Drive copy:', NOTEBOOK_DRIVE_COPY)
print(WORK_DIR)


In [ ]:
import os
from google.colab import userdata

for secret_name in ('DEEPSEEK_API_KEY', 'OPENAI_API_KEY', 'OPENAI_BASE_URL', 'OPENAI_MODEL'):
    if not os.environ.get(secret_name):
        try:
            secret_value = userdata.get(secret_name)
        except Exception:
            secret_value = None
        if secret_value:
            os.environ[secret_name] = secret_value
os.environ['DEEPSEEK_BASE_URL'] = os.environ.get('DEEPSEEK_BASE_URL', 'https://api.deepseek.com')
os.environ['OPENAI_MODEL'] = os.environ.get('OPENAI_MODEL', 'gpt-5.4')
os.environ['HF_HOME'] = f'{CLOUD_ROOT}/.cache/huggingface'
os.environ['TRANSFORMERS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/transformers'
os.environ['HF_DATASETS_CACHE'] = f'{CLOUD_ROOT}/.cache/huggingface/datasets'
os.environ['TORCH_HOME'] = f'{CLOUD_ROOT}/.cache/torch'
os.environ['XDG_CACHE_HOME'] = f'{CLOUD_ROOT}/.cache'
os.environ['WANDB_DIR'] = f'{CLOUD_ROOT}/wandb'
os.environ['TMPDIR'] = f'{CLOUD_ROOT}/tmp'
for key in ('HF_HOME', 'TRANSFORMERS_CACHE', 'HF_DATASETS_CACHE', 'TORCH_HOME', 'XDG_CACHE_HOME', 'WANDB_DIR', 'TMPDIR'):
    os.makedirs(os.environ[key], exist_ok=True)


In [ ]:
run_checked('pip-install', ['python', '-m', 'pip', 'install', '-q', '-r', 'requirements-cloud.txt'])


In [ ]:
run_checked('provider-probe', ['python', 'scripts/provider_probe.py', '--provider', 'colab', '--cloud-root', CLOUD_ROOT, '--report', f'{CLOUD_ROOT}/reports/platform_status_{RUN_ID}.json'])


In [ ]:
readiness_command = [
    'python',
    'scripts/colab_readiness_report.py',
    '--cloud-root', CLOUD_ROOT,
    '--run-id', RUN_ID,
    '--gp4-ws', GP4_WS,
    '--previous-adapter', os.environ['GP4_PREVIOUS_ADAPTER'],
    '--expected-commit', GP4_WS_EXPECTED_COMMIT,
    '--report', f'{CLOUD_ROOT}/reports/colab_readiness_{RUN_ID}.json',
]
if os.environ.get('GP4_OLD_DATASET'):
    readiness_command.extend(['--old-dataset', os.environ['GP4_OLD_DATASET']])
    readiness_command.append('--allow-mixed-prior-artifacts')
else:
    readiness_command.append('--allow-adapter-only-reuse')
run_checked('colab-readiness', readiness_command)

orchestrator_command = [
    'python',
    'scripts/cloud_orchestrator.py',
    '--run-id', RUN_ID,
    '--cloud-root', CLOUD_ROOT,
    '--seed', SEED_PATH,
    '--source-plan', SOURCE_PLAN,
    '--phases', 'ignored',
    '--preset', 'v2-300k',
]
if os.environ.get('GP4_OLD_DATASET'):
    orchestrator_command.extend(['--old-dataset', os.environ['GP4_OLD_DATASET']])
if os.environ.get('GP4_PREVIOUS_ADAPTER'):
    orchestrator_command.extend(['--previous-adapter', os.environ['GP4_PREVIOUS_ADAPTER']])
run_checked('cloud-orchestrator', orchestrator_command)


In [ ]:
run_checked('completion-audit', ['python', 'scripts/audit_cloud_completion.py', '--cloud-root', CLOUD_ROOT, '--run-id', RUN_ID, '--report', f'{CLOUD_ROOT}/reports/completion_audit_{RUN_ID}.json'])


The orchestrator stops automatically unless the previous gate report has `passed=true`. Completion evidence is `$CLOUD_ROOT/reports/acceptance_gate_report_$RUN_ID.json` with `passed=true`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.html`, `$CLOUD_ROOT/reports/benchmark_report_${RUN_ID}.md` with Maintenance Reference, and package metadata under the same cloud root.
